# Setup

In [ ]:
import shutil, os
if os.path.exists('e_network_inequality'):
    shutil.rmtree('e_network_inequality')

# !git clone https://github.com/IgnacioOQ/e_network_inequality
!git clone -b main https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
!pip install dill

In [ ]:
%cd e_network_inequality

/content/e_network_inequality/e_network_inequality


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from utils.imports import *
from model.agents import BetaAgent, BayesAgent
from model.model import Model
from utils.network_utils import *
from networks.network_generation import *
from networks.variation_methods import *
from model.simulation_functions import *
from model.vectorized_simulation_functions import *
from functools import partial
import hashlib
import gc
from multiprocessing import get_context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dumping_path = '/content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/'
print("Current Directory:", dumping_path)

Mounted at /content/drive
Current Directory: /content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/


In [ ]:
def generate_parameters_here(_,G,method='randomization'):
    process_seed = int.from_bytes(os.urandom(4), byteorder='little')
    rd.seed(process_seed)
    # Randomly sample parameters for this group
    uncertainty = rd.uniform(.000001, .001)
    n_experiments = rd.randint(1000, 10000)
    # now we pick a random number
    # Capped at 1/3 to prevent "Sample larger than population" errors in equalize
    proportion_edges = rd.rand() * (1/3)
    # Do randomization
    num_edges = G.number_of_edges()
    if method == 'randomization':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = randomize_network(G, n_edges=num_edges_to_randomize)
    if method == 'equalize':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = equalize(G, num_edges_to_randomize)
    if method == 'densify':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='original',keep_density_fixed=False)
    if method == 'densify_fixed':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='uniform',keep_density_fixed=True)
    if method =='cluster':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = cluster_network(G,num_edges_to_add)
    if method =='decluster':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = decluster_network(G,num_edges_to_randomize)

    result = generate_parameters_aggregate(modified_network, uncertainty=uncertainty, n_experiments=n_experiments,
                                           p_rewiring=proportion_edges)
    result['uncertainty'] = uncertainty
    result['n_experiments'] = n_experiments
    result['proportion_edges'] = proportion_edges
    result['parameter_random_seed']= process_seed
    return result

# Study 0: Load Networks

In [ ]:
# ── Load Networks + Parallelization Setup ────────────────────────────────────
from model.vectorized_model import VectorizedModel
from model.vectorized_simulation_functions import run_vectorized_simulation_with_params
from functools import partial
from multiprocessing import Pool, cpu_count
import itertools
import pickle
import networkx as nx

num_cores = cpu_count()
print(f"Available CPU cores: {num_cores}")

with open('./networks/citation_data/pud_network.pkl', 'rb') as f:
    G_pud = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_pud.nodes())}
G_pud_indexed = nx.relabel_nodes(G_pud, mapping)
print(f"PUD network: {G_pud_indexed.number_of_nodes()} nodes, {G_pud_indexed.number_of_edges()} edges")

with open('./networks/citation_data/tobacco_network.pkl', 'rb') as f:
    G_tobacco = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_tobacco.nodes())}
G_tobacco_indexed = nx.relabel_nodes(G_tobacco, mapping)
print(f"Tobacco network: {G_tobacco_indexed.number_of_nodes()} nodes, {G_tobacco_indexed.number_of_edges()} edges")

# Study: Compute Time and Truth Share Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from functools import partial
from multiprocessing import Pool
from tqdm.notebook import tqdm

N_EXPERIMENTS = 100
UNCERTAINTY = 0.0001
N_RUNS = 100
MAX_STEPS = 100_000


In [ ]:
def run_compute_time_study(network, network_label, output_prefix):
    print(f"=== Compute Time Study: {network_label} Network ===")
    
    param_dicts = [
        {"network": network,
         "n_experiments": N_EXPERIMENTS,
         "uncertainty": UNCERTAINTY,
         "seed": seed}
        for seed in range(N_RUNS)
    ]
    
    wrapper = partial(
        run_vectorized_simulation_with_params,
        tolerance=1e-5,
        tolerance_stopping=False,
        tstep_stopping=True,
        number_of_steps=MAX_STEPS,
        show_bar=False,
        agent_type="beta",
    )
    
    start_time = time.time()
    
    with Pool(num_cores) as pool:
        results = list(tqdm(
            pool.imap_unordered(wrapper, param_dicts),
            total=N_RUNS,
            desc=f"[{network_label}] Running simulations",
        ))
        
    end_time = time.time()
    compute_time = end_time - start_time
    print(f"  [{network_label}] Total compute time for {N_RUNS} runs: {compute_time:.2f} seconds")
    
    truth_shares = [r.get("share_of_correct_agents_at_convergence", 0) for r in results]
    mean_truth = np.mean(truth_shares)
    std_truth = np.std(truth_shares)
    
    print(f"  [{network_label}] Mean Truth Share: {mean_truth:.4f} (Std: {std_truth:.4f})")
    
    # Save results
    rows = [{"run": i, "truth_share": ts} for i, ts in enumerate(truth_shares)]
    df = pd.DataFrame(rows)
    df.to_csv(dumping_path + f"{output_prefix}_compute_time_results.csv", index=False)
    
    return compute_time, mean_truth, std_truth, truth_shares


In [ ]:
pud_time, pud_mean, pud_std, pud_shares = run_compute_time_study(G_pud_indexed, "PUD", "pud")
tobacco_time, tobacco_mean, tobacco_std, tobacco_shares = run_compute_time_study(G_tobacco_indexed, "Tobacco", "tobacco")


In [ ]:
networks = ["PUD", "Tobacco"]
means = [pud_mean, tobacco_mean]
stds = [pud_std, tobacco_std]
times = [pud_time, tobacco_time]

fig, ax1 = plt.subplots(figsize=(8, 6))
bars = ax1.bar(networks, means, yerr=stds, capsize=10, color=['#2980b9', '#e74c3c'], alpha=0.8)
ax1.set_ylabel('Mean Truth Share', fontsize=12)
ax1.set_title('Mean Truth Share and Compute Time per Network', fontsize=14)
ax1.set_ylim(0, 1.1)  # slightly above 1 for text

for i, v in enumerate(means):
    ax1.text(i, v + stds[i] + 0.02, f"{v:.3f} ± {stds[i]:.3f}\nTime: {times[i]:.1f}s", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plot_path = dumping_path + "compute_time_study_results.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved to {plot_path}")


## Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

# Get current time in New York
nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')

# Print and log
print(f"✅ Disconnected from runtime at: {formatted_time}")

# Disconnect Colab runtime
display(Javascript('google.colab.kernel.disconnect()'))

✅ Disconnected from runtime at: 2025-10-02 11:18:40 EDT


<IPython.core.display.Javascript object>